# Baseline 85/15 và final external-test trên Colab T4

Notebook điều phối ba baseline **ResNet-50 → DenseNet-121 → ViT-B/16**.
Model được train trên 12.750 ảnh, chọn checkpoint bằng macro-AUROC trên 2.250
validation images, đồng thời tự tối ưu 14 decision threshold theo F1 validation.

## Chuẩn bị thủ công trên máy local

```powershell
Compress-Archive `
  -Path 'data/processed/consensus_v1_224' `
  -DestinationPath 'consensus_v1_224.zip' `
  -CompressionLevel Optimal `
  -Force

Compress-Archive `
  -Path 'data/processed/consensus_test_v1_224' `
  -DestinationPath 'consensus_test_v1_224.zip' `
  -CompressionLevel Optimal `
  -Force
```

Upload hai archive, giữ nguyên folder gốc trong ZIP, vào:

```text
MyDrive/DATH_CXR/datasets/consensus_v1_224.zip
MyDrive/DATH_CXR/datasets/consensus_test_v1_224.zip
```

Ground truth CSV/JSON được clone từ nhánh `update_data`; ảnh, NPZ và ZIP không
nằm trong Git. Chạy lần đầu với `RUN_FINAL_TEST=False`. Sau khi ba checkpoint
đã được khóa, đổi thành `True` và Run All lại để đánh giá external test một lần.
Notebook không commit hoặc push.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/HuyDuongQuoc/DATH-HK261-explainable-cxr.git"
REPO_BRANCH = "update_data"
REPO_DIR = Path("/content/DATH-HK261-explainable-cxr")

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT / "MyDrive" / "DATH_CXR"
TRAIN_ARCHIVE = DRIVE_ROOT / "datasets" / "consensus_v1_224.zip"
TEST_ARCHIVE = DRIVE_ROOT / "datasets" / "consensus_test_v1_224.zip"
CHECKPOINT_ROOT = DRIVE_ROOT / "checkpoints"
RUN_LOG_ROOT = DRIVE_ROOT / "run_logs"

LOCAL_DATA_ROOT = Path("/content/datasets")
TRAIN_PROCESSED_ROOT = LOCAL_DATA_ROOT / "consensus_v1_224"
TEST_PROCESSED_ROOT = LOCAL_DATA_ROOT / "consensus_test_v1_224"
PROTOCOL_ROOT = REPO_DIR / "data" / "protocols" / "vindr_s8515_exttest_v1"
RUNTIME_CONFIG_ROOT = Path("/content/runtime_configs")
EXPORT_PATH = Path("/content/exports/02_experiment_results.zip")

REQUIRE_T4 = True
RUN_FINAL_TEST = False
EXPECTED_IMAGES = 15_000
EXPECTED_MASKS = 15_000
EXPECTED_TRAIN_ROWS = 12_750
EXPECTED_VAL_ROWS = 2_250
EXPECTED_TEST_IMAGES = 3_000
THRESHOLD_STRATEGY = "per_class_max_f1"

EXPERIMENTS = [
    {
        "run_name": "resnet50_baseline_s8515_exttest_v1_seed42_amp",
        "model_name": "resnet50",
        "config": "configs/resnet50_baseline.yaml",
    },
    {
        "run_name": "densenet121_baseline_s8515_exttest_v1_seed42_amp",
        "model_name": "densenet121",
        "config": "configs/densenet121_baseline.yaml",
    },
    {
        "run_name": "vit_baseline_s8515_exttest_v1_seed42_amp",
        "model_name": "vit_b_16",
        "config": "configs/vit_baseline.yaml",
    },
]

print("Experiments:", [item["run_name"] for item in EXPERIMENTS])
print("Final external test enabled:", RUN_FINAL_TEST)


## 1. Mount Drive và kiểm tra archive


In [ ]:
from google.colab import drive

drive.mount(str(DRIVE_MOUNT), force_remount=False)

if not TRAIN_ARCHIVE.is_file():
    raise FileNotFoundError(f"Thiếu train archive: {TRAIN_ARCHIVE}")
if RUN_FINAL_TEST and not TEST_ARCHIVE.is_file():
    raise FileNotFoundError(f"Thiếu external-test archive: {TEST_ARCHIVE}")

CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_LOG_ROOT.mkdir(parents=True, exist_ok=True)
print("Train archive:", TRAIN_ARCHIVE)
print("Test archive:", TEST_ARCHIVE if TEST_ARCHIVE.is_file() else "not uploaded")


## 2. Clone hoặc cập nhật source từ nhánh `update_data`


In [ ]:
import base64
import os
import subprocess
import sys
from getpass import getpass

github_token = getpass("GitHub PAT (để trống nếu repo public): ").strip()
git_auth_env = os.environ.copy()
if github_token:
    auth = base64.b64encode(
        f"x-access-token:{github_token}".encode("utf-8")
    ).decode("ascii")
    git_auth_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.extraHeader",
        "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}",
    })

def run_git(arguments, cwd=None):
    return subprocess.run(
        ["git", *arguments],
        cwd=cwd,
        env=git_auth_env,
        check=True,
    )

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} tồn tại nhưng không phải Git repository")
    run_git(["fetch", "origin", REPO_BRANCH], cwd=REPO_DIR)
    run_git(["checkout", REPO_BRANCH], cwd=REPO_DIR)
    run_git(["pull", "--ff-only", "origin", REPO_BRANCH], cwd=REPO_DIR)
else:
    run_git([
        "clone", "--branch", REPO_BRANCH, "--single-branch",
        REPO_URL, str(REPO_DIR),
    ])

os.chdir(REPO_DIR)
repo_path = str(REPO_DIR.resolve())
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

github_token = None
auth = None
git_auth_env = None
commit_sha = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print("Repository commit:", commit_sha)


## 3. Cài dependency và xác nhận GPU T4/AMP


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-common.txt")],
    check=True,
)

import torch
import torchvision

nvidia_smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    check=False,
    capture_output=True,
    text=True,
)
if nvidia_smi.returncode != 0 or not torch.cuda.is_available():
    raise RuntimeError("Colab runtime không có CUDA GPU")
gpu_name = torch.cuda.get_device_name(0)
if REQUIRE_T4 and "T4" not in gpu_name.upper():
    raise RuntimeError(f"Runtime đang dùng {gpu_name}, không phải Tesla T4")
with torch.autocast(device_type="cuda", dtype=torch.float16):
    amp_probe = torch.ones(8, 8, device="cuda") @ torch.ones(8, 8, device="cuda")
if not torch.isfinite(amp_probe).all():
    raise RuntimeError("AMP FP16 probe produced non-finite values")
del amp_probe

print(nvidia_smi.stdout.strip())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("AMP FP16 probe: OK")


## 4. Giải nén và xác minh train/validation/external-test artifacts


In [ ]:
import json
import shutil
import tempfile
import zipfile

import numpy as np
import pandas as pd

from src.data.protocol_splitting import assignment_sha256
from src.utils.checksums import file_sha256

REQUIRED_REPO_FILES = [
    REPO_DIR / "data" / "raw" / "test.csv",
    PROTOCOL_ROOT / "train_manifest.csv",
    PROTOCOL_ROOT / "val_manifest.csv",
    PROTOCOL_ROOT / "external_test_manifest.csv",
    PROTOCOL_ROOT / "split_metadata.json",
    PROTOCOL_ROOT / "external_test_metadata.json",
    PROTOCOL_ROOT / "checksums.json",
]
missing_repo_files = [str(path) for path in REQUIRED_REPO_FILES if not path.is_file()]
if missing_repo_files:
    raise FileNotFoundError(f"Clone thiếu ground-truth/protocol files: {missing_repo_files}")

def safe_extract_zip(archive_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            try:
                target.relative_to(destination)
            except ValueError as error:
                raise ValueError(f"Unsafe ZIP member: {member.filename}") from error
        archive.extractall(destination)

def ensure_extracted(archive_path, destination, folder_name, validator):
    if destination.exists():
        validator(destination)
        print("Dataset đã tồn tại và hợp lệ:", destination)
        return
    temporary_root = Path(tempfile.mkdtemp(prefix="cxr-extract-", dir="/content"))
    try:
        safe_extract_zip(archive_path, temporary_root)
        extracted_root = temporary_root / folder_name
        if not extracted_root.is_dir():
            raise FileNotFoundError(
                f"Archive phải chứa folder gốc {folder_name}/"
            )
        validator(extracted_root)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(extracted_root), str(destination))
    finally:
        shutil.rmtree(temporary_root, ignore_errors=True)

def validate_train_dataset(root):
    required = [root / "images", root / "masks", root / "manifest.csv", root / "metadata.json"]
    if not all(path.exists() for path in required):
        raise FileNotFoundError(f"Train processed root không đầy đủ: {root}")
    train = pd.read_csv(PROTOCOL_ROOT / "train_manifest.csv", dtype={"image_id": str})
    val = pd.read_csv(PROTOCOL_ROOT / "val_manifest.csv", dtype={"image_id": str})
    manifest = pd.read_csv(root / "manifest.csv", dtype={"image_id": str})
    if len(train) != EXPECTED_TRAIN_ROWS or len(val) != EXPECTED_VAL_ROWS:
        raise ValueError("Train/validation manifest count mismatch")
    train_ids, val_ids = set(train.image_id), set(val.image_id)
    if not train_ids.isdisjoint(val_ids):
        raise ValueError("Train/validation overlap")
    if train_ids | val_ids != set(manifest.image_id):
        raise ValueError("Train/validation không cover processed manifest")
    split_metadata = json.loads(
        (PROTOCOL_ROOT / "split_metadata.json").read_text(encoding="utf-8")
    )
    if assignment_sha256({"train": train, "val": val}) != split_metadata["assignment_sha256"]:
        raise ValueError("Split assignment fingerprint mismatch")
    if len(list((root / "images").glob("*.png"))) != EXPECTED_IMAGES:
        raise ValueError("Unexpected train PNG count")
    if len(list((root / "masks").glob("*.npz"))) != EXPECTED_MASKS:
        raise ValueError("Unexpected train NPZ count")
    return {
        "train_rows": len(train),
        "val_rows": len(val),
        "split_assignment_sha256": split_metadata["assignment_sha256"],
        "validation_manifest_sha256": file_sha256(PROTOCOL_ROOT / "val_manifest.csv"),
    }

def validate_test_dataset(root):
    required = [root / "images", root / "masks", root / "metadata.json"]
    if not all(path.exists() for path in required):
        raise FileNotFoundError(f"External-test processed root không đầy đủ: {root}")
    test = pd.read_csv(PROTOCOL_ROOT / "external_test_manifest.csv", dtype={"image_id": str})
    annotations = pd.read_csv(REPO_DIR / "data" / "raw" / "test.csv", dtype={"image_id": str})
    train = pd.read_csv(PROTOCOL_ROOT / "train_manifest.csv", dtype={"image_id": str})
    val = pd.read_csv(PROTOCOL_ROOT / "val_manifest.csv", dtype={"image_id": str})
    test_ids = set(test.image_id)
    image_ids = {path.stem for path in (root / "images").glob("*.png")}
    mask_ids = {path.stem for path in (root / "masks").glob("*.npz")}
    annotation_ids = set(annotations.image_id)
    if len(test) != EXPECTED_TEST_IMAGES or not test.image_id.is_unique:
        raise ValueError("External-test manifest count/uniqueness mismatch")
    if test_ids != image_ids or test_ids != mask_ids or test_ids != annotation_ids:
        raise ValueError("External-test CSV/PNG/NPZ ID mismatch")
    if test_ids & (set(train.image_id) | set(val.image_id)):
        raise ValueError("External test overlaps train/validation")
    metadata = json.loads(
        (PROTOCOL_ROOT / "external_test_metadata.json").read_text(encoding="utf-8")
    )
    if metadata["manifest_sha256"] != file_sha256(PROTOCOL_ROOT / "external_test_manifest.csv"):
        raise ValueError("External-test manifest hash mismatch")
    if metadata["annotation_sha256"] != file_sha256(REPO_DIR / "data" / "raw" / "test.csv"):
        raise ValueError("External-test annotation hash mismatch")
    findings = metadata["target_findings"]
    if not set(findings).issubset(test.columns):
        raise ValueError("External-test manifest thiếu finding columns")
    if not np.isin(test[findings].to_numpy(), [0, 1]).all():
        raise ValueError("External-test labels must be binary")
    return {"test_rows": len(test), "manifest_sha256": metadata["manifest_sha256"]}

ensure_extracted(
    TRAIN_ARCHIVE,
    TRAIN_PROCESSED_ROOT,
    "consensus_v1_224",
    validate_train_dataset,
)
train_summary = validate_train_dataset(TRAIN_PROCESSED_ROOT)

test_summary_data = None
if RUN_FINAL_TEST:
    ensure_extracted(
        TEST_ARCHIVE,
        TEST_PROCESSED_ROOT,
        "consensus_test_v1_224",
        validate_test_dataset,
    )
    test_summary_data = validate_test_dataset(TEST_PROCESSED_ROOT)

print("Train protocol:", train_summary)
print("External test:", test_summary_data or "locked")


## 5. Runtime config, resume và training ba baseline


In [ ]:
import yaml

from src.metrics.classification import THRESHOLD_STRATEGY as CODE_THRESHOLD_STRATEGY
from src.utils.config import load_config

if CODE_THRESHOLD_STRATEGY != THRESHOLD_STRATEGY:
    raise ValueError("Notebook/code threshold strategy mismatch")

def build_runtime_config(experiment):
    source_config_path = REPO_DIR / experiment["config"]
    config = load_config(source_config_path)
    config["data"]["processed_root"] = str(TRAIN_PROCESSED_ROOT)
    config["data"]["protocol_dir"] = str(PROTOCOL_ROOT)
    config["data"]["validation_ids_path"] = str(PROTOCOL_ROOT / "validation_ids.csv")
    config["data"]["train_manifest_path"] = str(PROTOCOL_ROOT / "train_manifest.csv")
    config["data"]["val_manifest_path"] = str(PROTOCOL_ROOT / "val_manifest.csv")
    config["data"]["external_test"]["annotation_csv"] = str(REPO_DIR / "data" / "raw" / "test.csv")
    config["data"]["external_test"]["processed_root"] = str(TEST_PROCESSED_ROOT)
    config["data"]["external_test"]["manifest_path"] = str(
        PROTOCOL_ROOT / "external_test_manifest.csv"
    )

    runtime_dir = RUNTIME_CONFIG_ROOT / experiment["run_name"]
    runtime_dir.mkdir(parents=True, exist_ok=True)
    runtime_config_path = runtime_dir / "default.yaml"
    runtime_config_path.write_text(
        yaml.safe_dump(config, sort_keys=False, allow_unicode=True),
        encoding="utf-8",
    )
    return runtime_config_path, config

def validate_threshold_checkpoint(checkpoint, config, checkpoint_path):
    findings = config["data"]["target_findings"]
    thresholds = checkpoint.get("decision_thresholds")
    if checkpoint.get("threshold_strategy") != THRESHOLD_STRATEGY:
        raise ValueError(f"Threshold strategy mismatch: {checkpoint_path}")
    if not isinstance(thresholds, dict) or list(thresholds) != findings:
        raise ValueError(f"Checkpoint threshold/class order mismatch: {checkpoint_path}")
    values = np.asarray(list(thresholds.values()), dtype=float)
    if not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():
        raise ValueError(f"Invalid checkpoint thresholds: {checkpoint_path}")
    if checkpoint.get("threshold_validation_manifest_sha256") != train_summary["validation_manifest_sha256"]:
        raise ValueError(f"Threshold validation manifest mismatch: {checkpoint_path}")

def inspect_run(experiment):
    config_path, config = build_runtime_config(experiment)
    checkpoint_dir = CHECKPOINT_ROOT / experiment["run_name"]
    result_dir = RUN_LOG_ROOT / experiment["run_name"]
    last_checkpoint = checkpoint_dir / "last.pt"
    state = {
        "config_path": config_path,
        "config": config,
        "checkpoint_dir": checkpoint_dir,
        "result_dir": result_dir,
        "last_checkpoint": last_checkpoint,
        "resume": False,
        "complete": False,
    }
    if last_checkpoint.is_file():
        checkpoint = torch.load(last_checkpoint, map_location="cpu", weights_only=False)
        if checkpoint.get("model_name") != experiment["model_name"]:
            raise ValueError(f"Checkpoint backbone mismatch: {last_checkpoint}")
        if checkpoint.get("split_assignment_sha256") != train_summary["split_assignment_sha256"]:
            raise ValueError(f"Checkpoint split fingerprint mismatch: {last_checkpoint}")
        if checkpoint.get("data_protocol_id") != "vindr_s8515_exttest_v1":
            raise ValueError(f"Checkpoint data protocol mismatch: {last_checkpoint}")
        validate_threshold_checkpoint(checkpoint, config, last_checkpoint)
        completed_epochs = int(checkpoint["epoch"]) + 1
        reached_limit = completed_epochs >= int(config["training"]["epochs"])
        early_stopped = int(checkpoint["bad_epochs"]) >= int(config["training"]["patience"])
        state.update({
            "complete": reached_limit or early_stopped,
            "resume": not (reached_limit or early_stopped),
            "completed_epochs": completed_epochs,
        })
    return state

def train_experiment(experiment):
    state = inspect_run(experiment)
    state["checkpoint_dir"].mkdir(parents=True, exist_ok=True)
    state["result_dir"].mkdir(parents=True, exist_ok=True)
    if state["complete"]:
        print(f"[{experiment['run_name']}] training đã hoàn tất; skip.")
    else:
        command = [
            sys.executable,
            str(REPO_DIR / "scripts" / "train.py"),
            "--config", str(state["config_path"]),
            "--checkpoint-dir", str(state["checkpoint_dir"]),
            "--result-dir", str(state["result_dir"]),
        ]
        if state["resume"]:
            command.extend(["--resume", str(state["last_checkpoint"])])
        training_env = os.environ.copy()
        training_env["PYTHONPATH"] = str(REPO_DIR)
        subprocess.run(command, cwd=REPO_DIR, env=training_env, check=True)
    best_checkpoint = state["checkpoint_dir"] / "best.pt"
    if not best_checkpoint.is_file():
        raise FileNotFoundError(best_checkpoint)
    return inspect_run(experiment)


In [ ]:
best_metrics_by_run = {}
for index, experiment in enumerate(EXPERIMENTS, start=1):
    print(f"[{index}/{len(EXPERIMENTS)}] {experiment['run_name']}")
    state = train_experiment(experiment)
    checkpoint_path = state["checkpoint_dir"] / "best.pt"
    metrics_path = state["result_dir"] / "best_val_metrics.json"
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    validate_threshold_checkpoint(checkpoint, state["config"], checkpoint_path)
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    if metrics.get("decision_thresholds") != checkpoint["decision_thresholds"]:
        raise ValueError(f"Best metrics/checkpoint threshold mismatch: {checkpoint_path}")
    best_metrics_by_run[experiment["run_name"]] = metrics
    print(
        f"best epoch={metrics['checkpoint_epoch']}, "
        f"macro-AUROC={metrics['macro_auroc']:.4f}, "
        f"tuned macro-F1={metrics['macro_f1']:.4f}"
    )

print("All training/validation experiments completed.")


## 6. Khóa checkpoint và final external test

Phần này chỉ chạy khi `RUN_FINAL_TEST=True`. Threshold không được tối ưu lại trên
test; `scripts/evaluate.py` đọc 14 threshold đã calibration trên validation từ
từng `best.pt`.


In [ ]:
def build_final_test_plan():
    models = {}
    for experiment in EXPERIMENTS:
        run_name = experiment["run_name"]
        checkpoint_path = CHECKPOINT_ROOT / run_name / "best.pt"
        metrics_path = RUN_LOG_ROOT / run_name / "best_val_metrics.json"
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        config_path, config = build_runtime_config(experiment)
        validate_threshold_checkpoint(checkpoint, config, checkpoint_path)
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        models[run_name] = {
            "model_name": experiment["model_name"],
            "checkpoint": str(checkpoint_path),
            "checkpoint_sha256": file_sha256(checkpoint_path),
            "best_epoch": int(checkpoint["epoch"]) + 1,
            "validation_macro_auroc": float(metrics["macro_auroc"]),
            "validation_macro_f1_tuned": float(metrics["macro_f1"]),
            "decision_thresholds": checkpoint["decision_thresholds"],
        }
    return {
        "data_protocol_id": "vindr_s8515_exttest_v1",
        "split_assignment_sha256": train_summary["split_assignment_sha256"],
        "validation_manifest_sha256": train_summary["validation_manifest_sha256"],
        "threshold_strategy": THRESHOLD_STRATEGY,
        "models": models,
    }

def verify_test_outputs(experiment, plan_entry, output_dir):
    metrics_path = output_dir / "test_metrics.json"
    predictions_path = output_dir / "test_predictions.csv"
    metadata_path = output_dir / "evaluation_metadata.json"
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    predictions = pd.read_csv(predictions_path, dtype={"image_id": str})
    manifest = pd.read_csv(
        PROTOCOL_ROOT / "external_test_manifest.csv", dtype={"image_id": str}
    )
    findings = metadata["findings"]
    if int(metrics["num_images"]) != EXPECTED_TEST_IMAGES or len(predictions) != EXPECTED_TEST_IMAGES:
        raise ValueError(f"[{experiment['run_name']}] external-test row count mismatch")
    if not predictions.image_id.is_unique or set(predictions.image_id) != set(manifest.image_id):
        raise ValueError(f"[{experiment['run_name']}] prediction image IDs mismatch")
    expected_columns = ["image_id"]
    for finding in findings:
        expected_columns.extend([
            f"target::{finding}",
            f"probability::{finding}",
            f"prediction::{finding}",
        ])
    if list(predictions.columns) != expected_columns:
        raise ValueError(f"[{experiment['run_name']}] prediction columns mismatch")
    probability_columns = [f"probability::{finding}" for finding in findings]
    prediction_columns = [f"prediction::{finding}" for finding in findings]
    if not np.isfinite(predictions[probability_columns].to_numpy(dtype=float)).all():
        raise ValueError("Non-finite external-test probability")
    if not np.isin(predictions[prediction_columns].to_numpy(), [0, 1]).all():
        raise ValueError("Non-binary external-test prediction")
    by_id = predictions.set_index("image_id")
    manifest_by_id = manifest.set_index("image_id")
    for finding in findings:
        if not np.array_equal(
            by_id.loc[manifest_by_id.index, f"target::{finding}"].to_numpy(dtype=np.uint8),
            manifest_by_id[finding].to_numpy(dtype=np.uint8),
        ):
            raise ValueError(f"Target mismatch for {finding}")
    if metadata["threshold_source"] != "checkpoint":
        raise ValueError("External test did not use checkpoint thresholds")
    if metadata["threshold_strategy"] != THRESHOLD_STRATEGY:
        raise ValueError("External-test threshold strategy mismatch")
    if metadata["decision_thresholds"] != plan_entry["decision_thresholds"]:
        raise ValueError("External-test threshold mapping mismatch")
    if metadata["checkpoint_sha256"] != plan_entry["checkpoint_sha256"]:
        raise ValueError("External-test checkpoint hash mismatch")
    return metrics, metadata

final_test_summary = None
if not RUN_FINAL_TEST:
    print("External test is disabled. Set RUN_FINAL_TEST=True after checkpoint selection.")
else:
    validate_test_dataset(TEST_PROCESSED_ROOT)
    final_test_plan_path = RUN_LOG_ROOT / "final_test_plan.json"
    requested_plan = build_final_test_plan()
    if final_test_plan_path.exists():
        stored_plan = json.loads(final_test_plan_path.read_text(encoding="utf-8"))
        if stored_plan != requested_plan:
            raise RuntimeError("Existing final_test_plan.json does not match locked checkpoints")
    else:
        final_test_plan_path.write_text(
            json.dumps(requested_plan, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

    rows = []
    for experiment in EXPERIMENTS:
        run_name = experiment["run_name"]
        config_path, _ = build_runtime_config(experiment)
        checkpoint_path = CHECKPOINT_ROOT / run_name / "best.pt"
        output_dir = RUN_LOG_ROOT / run_name / "final_test"
        expected_outputs = [
            output_dir / "test_metrics.json",
            output_dir / "test_predictions.csv",
            output_dir / "evaluation_metadata.json",
        ]
        existing = [path for path in expected_outputs if path.exists()]
        if existing and len(existing) != len(expected_outputs):
            raise RuntimeError(f"[{run_name}] partial final-test artifacts: {existing}")
        if not existing:
            test_env = os.environ.copy()
            test_env["PYTHONPATH"] = str(REPO_DIR)
            subprocess.run(
                [
                    sys.executable,
                    str(REPO_DIR / "scripts" / "evaluate.py"),
                    "--config", str(config_path),
                    "--checkpoint", str(checkpoint_path),
                    "--output-dir", str(output_dir),
                    "--split", "test",
                ],
                cwd=REPO_DIR,
                env=test_env,
                check=True,
            )
        metrics, metadata = verify_test_outputs(
            experiment,
            requested_plan["models"][run_name],
            output_dir,
        )
        rows.append({
            "run_name": run_name,
            "model": experiment["model_name"],
            "checkpoint_epoch": metadata["checkpoint_epoch"],
            "macro_auroc": metrics["macro_auroc"],
            "macro_average_precision": metrics["macro_average_precision"],
            "macro_precision": metrics["macro_precision"],
            "macro_recall": metrics["macro_recall"],
            "macro_f1": metrics["macro_f1"],
            "elapsed_seconds": metrics["elapsed_seconds"],
            "images_per_second": metrics["images_per_second"],
            "checkpoint_sha256": metadata["checkpoint_sha256"],
        })
    final_test_summary = pd.DataFrame(rows)
    final_test_summary.to_csv(
        RUN_LOG_ROOT / "external_test_baseline_summary.csv", index=False
    )
    display(final_test_summary)


## 7. Phân tích và tạo figure external test


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_curve, auc

def build_external_test_figures():
    figure_root = RUN_LOG_ROOT / "external_test_figures"
    figure_root.mkdir(parents=True, exist_ok=True)
    model_labels = {
        "resnet50": "ResNet-50",
        "densenet121": "DenseNet-121",
        "vit_b_16": "ViT-B/16",
    }
    model_order = ["ResNet-50", "DenseNet-121", "ViT-B/16"]
    colors = {"ResNet-50": "#4C78A8", "DenseNet-121": "#F58518", "ViT-B/16": "#54A24B"}
    summary_rows, class_frames, test_data = [], [], {}
    reference_ids, reference_targets, finding_order = None, None, None

    for experiment in EXPERIMENTS:
        model_label = model_labels[experiment["model_name"]]
        test_dir = RUN_LOG_ROOT / experiment["run_name"] / "final_test"
        metrics = json.loads((test_dir / "test_metrics.json").read_text(encoding="utf-8"))
        metadata = json.loads((test_dir / "evaluation_metadata.json").read_text(encoding="utf-8"))
        predictions = pd.read_csv(test_dir / "test_predictions.csv")
        findings = metadata["findings"]
        finding_order = findings if finding_order is None else finding_order
        target_columns = [f"target::{finding}" for finding in findings]
        probability_columns = [f"probability::{finding}" for finding in findings]
        image_ids = predictions.image_id.astype(str).to_numpy()
        targets = predictions[target_columns].to_numpy(dtype=np.uint8)
        probabilities = predictions[probability_columns].to_numpy(dtype=float)
        if reference_ids is None:
            reference_ids, reference_targets = image_ids, targets
        elif not np.array_equal(image_ids, reference_ids) or not np.array_equal(targets, reference_targets):
            raise ValueError("External-test predictions are not aligned across models")
        test_data[model_label] = {"targets": targets, "probabilities": probabilities}
        summary_rows.append({
            "Model": model_label,
            "Macro AUROC": metrics["macro_auroc"],
            "Macro AUPRC": metrics["macro_average_precision"],
            "Macro Precision": metrics["macro_precision"],
            "Macro Recall": metrics["macro_recall"],
            "Macro F1": metrics["macro_f1"],
            "Elapsed seconds": metrics["elapsed_seconds"],
            "Images/second": metrics["images_per_second"],
            "Peak GPU memory (GB)": metrics["peak_gpu_memory_gb"],
        })
        class_frame = pd.DataFrame(metrics["per_class"])
        class_frame["Model"] = model_label
        class_frames.append(class_frame)

    summary_df = pd.DataFrame(summary_rows).set_index("Model").reindex(model_order).reset_index()
    per_class_df = pd.concat(class_frames, ignore_index=True)
    summary_df.to_csv(figure_root / "external_test_model_summary.csv", index=False)
    per_class_df.to_csv(figure_root / "external_test_per_class_metrics.csv", index=False)
    sns.set_theme(style="whitegrid", context="talk")

    macro_columns = ["Macro AUROC", "Macro AUPRC", "Macro Precision", "Macro Recall", "Macro F1"]
    macro_long = summary_df.melt(id_vars="Model", value_vars=macro_columns, var_name="Metric", value_name="Score")
    fig, ax = plt.subplots(figsize=(15, 7))
    sns.barplot(data=macro_long, x="Metric", y="Score", hue="Model", hue_order=model_order, palette=colors, ax=ax)
    ax.set(title="External-test metrics — validation-tuned per-class thresholds", xlabel="", ylabel="Score", ylim=(0, 1))
    fig.tight_layout()
    fig.savefig(figure_root / "01_macro_metrics_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(20, 11), sharey=True)
    for ax, (column, title) in zip(axes, [("auroc", "AUROC"), ("average_precision", "AUPRC"), ("f1", "F1 — tuned thresholds")]):
        matrix = per_class_df.pivot(index="finding", columns="Model", values=column).reindex(index=finding_order, columns=model_order)
        sns.heatmap(matrix, annot=True, fmt=".3f", cmap="YlGnBu", vmin=0, vmax=1, ax=ax, cbar=ax is axes[-1])
        ax.set_title(title)
    fig.tight_layout()
    fig.savefig(figure_root / "02_per_class_metrics_heatmap.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    prevalence = reference_targets.mean()
    for label in model_order:
        targets = test_data[label]["targets"].ravel()
        probabilities = test_data[label]["probabilities"].ravel()
        fpr, tpr, _ = roc_curve(targets, probabilities)
        precision, recall, _ = precision_recall_curve(targets, probabilities)
        axes[0].plot(fpr, tpr, color=colors[label], label=f"{label} — AUC={auc(fpr, tpr):.3f}")
        axes[1].plot(recall, precision, color=colors[label], label=f"{label} — AP={average_precision_score(targets, probabilities):.3f}")
    axes[0].plot([0, 1], [0, 1], "--", color="gray")
    axes[1].axhline(prevalence, linestyle="--", color="gray")
    axes[0].set(title="External-test micro ROC", xlabel="FPR", ylabel="TPR")
    axes[1].set(title="External-test micro Precision–Recall", xlabel="Recall", ylabel="Precision")
    axes[0].legend(fontsize=9); axes[1].legend(fontsize=9)
    fig.tight_layout()
    fig.savefig(figure_root / "03_micro_roc_pr_curves.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(12, 8))
    sns.scatterplot(data=per_class_df, x="positive_count", y="average_precision", hue="Model", hue_order=model_order, palette=colors, s=110, ax=ax)
    ax.set_xscale("log")
    ax.set(title="External-test support vs AUPRC", xlabel="Positive support — log scale", ylabel="AUPRC", ylim=(0, 1.02))
    fig.tight_layout()
    fig.savefig(figure_root / "04_class_support_vs_auprc.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax, (column, title) in zip(axes, [("Images/second", "Throughput"), ("Elapsed seconds", "Evaluation time"), ("Peak GPU memory (GB)", "Peak GPU memory")]):
        sns.barplot(data=summary_df, x="Model", y=column, order=model_order, hue="Model", palette=colors, legend=False, ax=ax)
        ax.set_title(title); ax.tick_params(axis="x", rotation=20)
    fig.tight_layout()
    fig.savefig(figure_root / "05_evaluation_efficiency.png", dpi=300, bbox_inches="tight")
    plt.show()

    figure_manifest = {
        "split": "external_test",
        "num_images": EXPECTED_TEST_IMAGES,
        "threshold_strategy": THRESHOLD_STRATEGY,
        "models": model_order,
    }
    (figure_root / "figure_manifest.json").write_text(
        json.dumps(figure_manifest, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    return figure_root

external_figure_root = None
if RUN_FINAL_TEST:
    external_figure_root = build_external_test_figures()
    print("External-test figures:", external_figure_root)
else:
    print("Skip external-test figures because RUN_FINAL_TEST=False")


## 8. Đóng gói metrics, threshold metadata và figures


In [ ]:
from datetime import datetime, timezone

staging_root = Path(tempfile.mkdtemp(prefix="cxr-export-", dir="/content"))
try:
    metrics_root = staging_root / "outputs" / "metrics"
    metrics_root.mkdir(parents=True, exist_ok=True)
    for experiment in EXPERIMENTS:
        run_name = experiment["run_name"]
        source_dir = RUN_LOG_ROOT / run_name
        destination = metrics_root / run_name
        destination.mkdir(parents=True, exist_ok=True)
        for filename in ["config.json", "history.csv", "latest_val_metrics.json", "best_val_metrics.json"]:
            source = source_dir / filename
            if not source.is_file():
                raise FileNotFoundError(source)
            shutil.copy2(source, destination / filename)
        if RUN_FINAL_TEST:
            shutil.copytree(source_dir / "final_test", destination / "final_test")

    if RUN_FINAL_TEST:
        shutil.copy2(RUN_LOG_ROOT / "final_test_plan.json", metrics_root / "final_test_plan.json")
        shutil.copy2(
            RUN_LOG_ROOT / "external_test_baseline_summary.csv",
            metrics_root / "external_test_baseline_summary.csv",
        )
        shutil.copytree(external_figure_root, metrics_root / "external_test_figures")

    experiment_manifest = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "repository_url": REPO_URL,
        "repository_branch": REPO_BRANCH,
        "git_commit": commit_sha,
        "data_protocol_id": "vindr_s8515_exttest_v1",
        "split_assignment_sha256": train_summary["split_assignment_sha256"],
        "validation_manifest_sha256": train_summary["validation_manifest_sha256"],
        "threshold_strategy": THRESHOLD_STRATEGY,
        "final_test_included": RUN_FINAL_TEST,
        "train_archive": str(TRAIN_ARCHIVE),
        "train_archive_sha256": file_sha256(TRAIN_ARCHIVE),
        "test_archive": str(TEST_ARCHIVE) if RUN_FINAL_TEST else None,
        "test_archive_sha256": file_sha256(TEST_ARCHIVE) if RUN_FINAL_TEST else None,
        "checkpoint_included_in_export": False,
    }
    (metrics_root / "experiment_manifest.json").write_text(
        json.dumps(experiment_manifest, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    if EXPORT_PATH.exists():
        EXPORT_PATH.unlink()
    shutil.make_archive(str(EXPORT_PATH.with_suffix("")), "zip", root_dir=staging_root)
    with zipfile.ZipFile(EXPORT_PATH) as archive:
        members = [name for name in archive.namelist() if not name.endswith("/")]
    forbidden = (".pt", ".pth", ".ckpt", ".npz")
    if any(name.lower().endswith(forbidden) for name in members):
        raise RuntimeError("Result archive chứa checkpoint/data binary ngoài ý muốn")
    print("Export ready:", EXPORT_PATH)
    print("Files in export:", len(members))
finally:
    shutil.rmtree(staging_root, ignore_errors=True)
